In [ ]:
!pip install datasets -q

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All good!")

In [ ]:
ds = load_dataset("surrey-nlp/BESSTIE-CW-26")

# Combine all splits into one DataFrame
frames = []
for split_name, split_data in ds.items():
    df_split = split_data.to_pandas()
    df_split['split'] = split_name
    frames.append(df_split)

df = pd.concat(frames, ignore_index=True)

print(f"Total rows: {len(df)}")
print(df['variety'].value_counts())

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['text'].astype(str).apply(clean_text)

print("Done! Sample:")
print(df[['variety', 'clean_text']].head(3).to_string(index=False))

In [ ]:
texts_uk = df[df['variety'] == 'en-UK']['clean_text'].tolist()
texts_in = df[df['variety'] == 'en-IN']['clean_text'].tolist()
texts_au = df[df['variety'] == 'en-AU']['clean_text'].tolist()

print(f"en-UK: {len(texts_uk)} texts")
print(f"en-IN: {len(texts_in)} texts")
print(f"en-AU: {len(texts_au)} texts")

In [ ]:
STOPWORDS = {
    'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'she', 'it', 'its',
    'they', 'their', 'them', 'this', 'that', 'these', 'those', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does',
    'did', 'will', 'would', 'could', 'should', 'may', 'might', 'shall',
    'a', 'an', 'the', 'and', 'but', 'or', 'for', 'yet', 'so',
    'at', 'by', 'in', 'of', 'on', 'to', 'up', 'as', 'if', 'with', 'not',
    'no', 'from', 'into', 'than', 'then', 'just', 'also', 'about',
    'what', 'which', 'who', 'when', 'where', 'how', 'all', 'each', 'more',
    'very', 'too', 'get', 'got', 'go', 'went', 'come', 'came', 'make',
}

def build_vocab(texts):
    vocab = set()
    for text in texts:
        for word in text.split():
            if len(word) >= 3 and word not in STOPWORDS:
                vocab.add(word)
    return vocab

vocab_uk = build_vocab(texts_uk)
vocab_in = build_vocab(texts_in)
vocab_au = build_vocab(texts_au)

print(f"en-UK vocabulary: {len(vocab_uk):,} unique words")
print(f"en-IN vocabulary: {len(vocab_in):,} unique words")
print(f"en-AU vocabulary: {len(vocab_au):,} unique words")

In [ ]:
def jaccard_similarity(set_a, set_b):
    intersection = set_a & set_b
    union = set_a | set_b
    return len(intersection) / len(union)

j_uk_in = jaccard_similarity(vocab_uk, vocab_in)
j_uk_au = jaccard_similarity(vocab_uk, vocab_au)
j_au_in = jaccard_similarity(vocab_au, vocab_in)

print(f"en-UK vs en-IN : {j_uk_in:.4f} ({j_uk_in*100:.2f}%)")
print(f"en-UK vs en-AU : {j_uk_au:.4f} ({j_uk_au*100:.2f}%)")
print(f"en-AU vs en-IN : {j_au_in:.4f} ({j_au_in*100:.2f}%)")

# Also show the raw word counts
shared = vocab_uk & vocab_in
uk_only = vocab_uk - vocab_in
in_only = vocab_in - vocab_uk

print(f"\nWords shared by en-UK and en-IN : {len(shared):,}")
print(f"Words only in en-UK             : {len(uk_only):,}")
print(f"Words only in en-IN             : {len(in_only):,}")

en-UK and en-AU share 34.87% of vocabulary — they are the most similar, which makes sense because both are "inner-circle" Western English varieties
en-UK and en-IN only share 29.26% — Indian English is noticeably more different from British English
en-AU and en-IN are the most different at 28.14%
Out of ~13,000 total words between UK and IN, only 4,314 are shared — meaning more than half the words are unique to one variety

In [ ]:
#TF-IDF Cosine Similarity
corpus_uk = ' '.join(texts_uk)
corpus_in = ' '.join(texts_in)
corpus_au = ' '.join(texts_au)

tfidf = TfidfVectorizer(max_features=20000, min_df=1, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform([corpus_uk, corpus_in, corpus_au])

cos_uk_in = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
cos_uk_au = cosine_similarity(tfidf_matrix[0], tfidf_matrix[2])[0][0]
cos_au_in = cosine_similarity(tfidf_matrix[2], tfidf_matrix[1])[0][0]

print(f"en-UK vs en-IN : {cos_uk_in:.4f}")
print(f"en-UK vs en-AU : {cos_uk_au:.4f}")
print(f"en-AU vs en-IN : {cos_au_in:.4f}")

In [ ]:
# Words unique to Indian English
from collections import Counter

def word_freq(texts):
    counter = Counter()
    for text in texts:
        for word in text.split():
            if len(word) >= 3 and word not in STOPWORDS:
                counter[word] += 1
    return counter

freq_uk = word_freq(texts_uk)
freq_in = word_freq(texts_in)

in_only = vocab_in - vocab_uk
uk_only = vocab_uk - vocab_in

# Most frequent words unique to Indian English
in_only_freq = {w: freq_in[w] for w in in_only if w in freq_in}
top_in = sorted(in_only_freq.items(), key=lambda x: x[1], reverse=True)[:20]

print("Most frequent words UNIQUE to en-IN (not seen in en-UK):")
print("-" * 40)
for word, count in top_in:
    print(f"  {word:<25} {count:>4} times")

print()

# Most frequent words unique to British English
uk_only_freq = {w: freq_uk[w] for w in uk_only if w in freq_uk}
top_uk = sorted(uk_only_freq.items(), key=lambda x: x[1], reverse=True)[:20]

print("Most frequent words UNIQUE to en-UK (not seen in en-IN):")
print("-" * 40)
for word, count in top_uk:
    print(f"  {word:<25} {count:>4} times")

Indian English unique words tell a clear story:

hai, toh, bhi, nahi, mein, kya, tha — these are Hindi words mixed into English sentences. This is code-mixing, a real linguistic phenomenon
biriyani, chutney, pav, wada, nonveg — Indian food culture
bjp, modi — Indian politics
temple, bollywood, rupees, punjabi — Indian cultural references

British English unique words:

tories, tory, nhs, wetherspoons, spoons, pint — very British pub and politics culture
vegan, whilst, popped — distinctly British vocabulary

In [ ]:
# Vocabulary overlap bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Vocabulary Composition by Variety Pair', fontsize=14, fontweight='bold')

pairs = [
    ('en-UK', 'en-IN', vocab_uk, vocab_in),
    ('en-UK', 'en-AU', vocab_uk, vocab_au),
    ('en-AU', 'en-IN', vocab_au, vocab_in),
]

jaccard_scores = {
    ('en-UK', 'en-IN'): j_uk_in,
    ('en-UK', 'en-AU'): j_uk_au,
    ('en-AU', 'en-IN'): j_au_in,
}

for ax, (label_a, label_b, va, vb) in zip(axes, pairs):
    shared  = len(va & vb)
    only_a  = len(va - vb)
    only_b  = len(vb - va)
    total   = only_a + shared + only_b

    categories = [f'{label_a} only', 'Shared', f'{label_b} only']
    values     = [only_a, shared, only_b]
    colours    = ['#FF9800', '#2196F3', '#F44336']

    bars = ax.bar(categories, values, color=colours, edgecolor='white', linewidth=1.2)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{val:,}\n({val/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

    j = jaccard_scores[(label_a, label_b)]
    ax.set_title(f'{label_a} vs {label_b}\nJaccard = {j:.4f}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Number of unique words')
    ax.set_ylim(0, max(values) * 1.3)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('vocab_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Similarity Heatmap
import numpy as np

varieties = ['en-UK', 'en-IN', 'en-AU']

jaccard_matrix = np.array([
    [1.0,      j_uk_in,  j_uk_au],
    [j_uk_in,  1.0,      j_au_in],
    [j_uk_au,  j_au_in,  1.0    ],
])

cosine_matrix = np.array([
    [1.0,        cos_uk_in,  cos_uk_au],
    [cos_uk_in,  1.0,        cos_au_in],
    [cos_uk_au,  cos_au_in,  1.0      ],
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

titles = [
    'Jaccard Similarity\n(raw vocabulary overlap)',
    'TF-IDF Cosine Similarity\n(importance-weighted overlap)'
]

for ax, matrix, title in zip(axes, [jaccard_matrix, cosine_matrix], titles):
    sns.heatmap(matrix, annot=True, fmt='.4f', cmap='YlOrRd',
                xticklabels=varieties, yticklabels=varieties,
                vmin=0, vmax=1, linewidths=0.5, ax=ax,
                annot_kws={'size': 12, 'weight': 'bold'})
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)

fig.suptitle('Vocabulary Similarity Between English Varieties (BESSTIE)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('vocab_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

 the TF-IDF scores (right heatmap) are much higher than Jaccard (left heatmap). For example UK vs IN jumps from 0.2926 to 0.5477. This tells you that the important, high-frequency words are actually more shared — it's the rare and niche words driving the difference.

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())

# Extended stopwords to filter out boring common words
TFIDF_STOPWORDS = {
    'the', 'and', 'to', 'was', 'of', 'for', 'is', 'it', 'in', 'with',
    'but', 'we', 'on', 'that', 'this', 'are', 'be', 'at', 'as', 'an',
    'have', 'had', 'has', 'not', 'they', 'he', 'she', 'you', 'his', 'her',
    'our', 'my', 'me', 'us', 'its', 'so', 'or', 'if', 'by', 'up', 'do',
    'did', 'got', 'get', 'ka', 'ke', 'rs', 'ki', 'ne', 'se', 'hai',
}

def get_top_words_filtered(row_index, n=15):
    scores = tfidf_matrix[row_index].toarray().flatten()
    # Sort all words by score descending
    sorted_idx = scores.argsort()[::-1]
    filtered_words = []
    filtered_scores = []
    for idx in sorted_idx:
        word = feature_names[idx]
        if word not in TFIDF_STOPWORDS and len(word) >= 3:
            filtered_words.append(word)
            filtered_scores.append(scores[idx])
        if len(filtered_words) == n:
            break
    # Reverse so highest is at top of chart
    return filtered_words[::-1], filtered_scores[::-1]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
colours = {'en-UK': '#1976D2', 'en-IN': '#D32F2F', 'en-AU': '#388E3C'}

for ax, (variety, row_idx) in zip(axes, [('en-UK', 0), ('en-IN', 1), ('en-AU', 2)]):
    words, scores = get_top_words_filtered(row_idx, n=15)
    ax.barh(words, scores, color=colours[variety], edgecolor='white', linewidth=0.8)
    ax.set_xlabel('TF-IDF Score', fontsize=10)
    ax.set_title(f'Most Distinctive Words\n{variety}',
                 fontsize=12, fontweight='bold', color=colours[variety])
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='y', labelsize=10)

fig.suptitle('Top TF-IDF Distinctive Words by English Variety',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()

en-UK: tories, wetherspoons, staff, service — British pub culture and politics
en-IN: toh, bhi, bjp, biriyani, nahi, temple — Hindi code-mixing and Indian cultural references
en-AU: labor, australian, melbourne, sydney, dutton, federal — Australian politics and cities

In [ ]:
print("=" * 55)
print("  Q1.2 SUMMARY — copy these into your report")
print("=" * 55)
print()
print("VOCABULARY SIZES:")
print(f"  en-UK : {len(vocab_uk):,} unique words")
print(f"  en-IN : {len(vocab_in):,} unique words")
print(f"  en-AU : {len(vocab_au):,} unique words")
print()
print("JACCARD SIMILARITY:")
print(f"  en-UK vs en-IN : {j_uk_in:.4f} ({j_uk_in*100:.2f}%)")
print(f"  en-UK vs en-AU : {j_uk_au:.4f} ({j_uk_au*100:.2f}%)")
print(f"  en-AU vs en-IN : {j_au_in:.4f} ({j_au_in*100:.2f}%)")
print()
print("TF-IDF COSINE SIMILARITY:")
print(f"  en-UK vs en-IN : {cos_uk_in:.4f}")
print(f"  en-UK vs en-AU : {cos_uk_au:.4f}")
print(f"  en-AU vs en-IN : {cos_au_in:.4f}")
print()
print("UK vs IN WORD BREAKDOWN:")
print(f"  Shared words   : {len(vocab_uk & vocab_in):,}")
print(f"  en-UK only     : {len(vocab_uk - vocab_in):,}")
print(f"  en-IN only     : {len(vocab_in - vocab_uk):,}")
print("=" * 55)

Vocabulary Analysis
To quantify the lexical distance between the inner-circle variety British English (en-UK) and the outer-circle variety Indian English (en-IN), we computed two complementary metrics: Jaccard Similarity and TF-IDF Cosine Similarity.
The en-UK corpus contains 9,701 unique words and en-IN contains 9,358 unique words. Of these, only 4,314 words are shared between the two varieties, meaning 5,387 words are exclusive to en-UK and 5,044 words are exclusive to en-IN. The Jaccard Similarity between en-UK and en-IN is 0.2926 (29.26%), compared to 0.3487 (34.87%) between en-UK and en-AU. This confirms that Indian English is lexically more distant from British English than Australian English is. TF-IDF Cosine Similarity, which weights words by their importance rather than treating all words equally, reinforces this pattern: en-UK vs en-IN scores 0.5477 compared to 0.6252 for en-UK vs en-AU. The higher cosine scores relative to Jaccard suggest that high-frequency important words are more commonly shared across varieties, while rare and culturally specific words drive the divergence.
Inspection of the words unique to en-IN reveals two distinct sources of divergence. First, Hindi code-mixed words such as toh, bhi, nahi, mein and tha appear frequently, representing genuine structural linguistic distance where speakers switch between Hindi and English mid-sentence. Second, culturally specific vocabulary such as biriyani, temple, bollywood, rupees and bjp reflects topic-domain distance driven by Indian cultural and political context rather than grammatical differences. By contrast, en-UK exclusive words such as tories, wetherspoons, pint and nhs reflect British cultural and political context.
Linguistic distance refers to the degree of structural, lexical and phonological divergence between two language varieties. In NLP sequence classification, greater linguistic distance can degrade model performance because a model trained on one variety may encounter out-of-vocabulary tokens and culturally grounded expressions it was never exposed to during training. However, not all of the divergence observed here represents deep grammatical difference. Some of it is topic-domain distance, where the two corpora discuss different subjects rather than using fundamentally different grammar. The Hindi code-mixing in en-IN however represents genuine structural divergence that a model trained purely on British English is unlikely to handle robustly, suggesting cross-variety performance gaps are expected and worth investigating in our experiments.
